<a href="https://www.kaggle.com/code/sriharikrishnants/final-of-food-101?scriptVersionId=246437996" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
#contains all the images
data_path = '/kaggle/input/food41/'

In [ ]:
class_file_path = os.path.join(data_path, 'meta', 'meta', 'classes.txt')
class_to_id = {}

with open(class_file_path, 'r') as f:
    # Read all lines ONCE and strip whitespace
    class_names = [line.strip() for line in f.readlines()]
    # Map index to class name
    class_to_id = {name:i for i, name in enumerate(class_names)}
    id_to_class = {i:name for i,name in enumerate(class_names)}
    

print('Label to Class:')
for idx, name in class_to_id.items():
    print(f'{idx} - {name}')

In [ ]:
#creating a dataset wrapper 
class Food101(Dataset):
    def __init__(self,data_path,file,transform):
        super().__init__()
        self.data_path = data_path
        self.file = file
        self.transform = transform
        self.image_paths = []
        self.labels = []

        with open(file,'r') as f:
            for line in f:
                rel = line.strip().split('/')[0]
                #if you use , it will add a slash but we want .jpg so use +
                path = os.path.join(data_path,'images',line.strip() + '.jpg')
                self.image_paths.append(path)
                self.labels.append(class_to_id[rel])
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self,idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')

        image = self.transform(image)

        return image,label

In [ ]:
train_path = os.path.join(data_path,'meta','meta','train.txt')
test_path = os.path.join(data_path,'meta','meta','test.txt')

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    # Randomly crop a section of the image and resize it to IMAGE_SIZE x IMAGE_SIZE.
    # The 'scale' argument (0.8, 1.0) means the cropped area will be between 80% and 100%
    # of the original image area. This makes the cropping less aggressive,
    # helping to retain more of the food item.
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),

    # Randomly flip the image horizontally. Good for augmenting food photos.
    transforms.RandomHorizontalFlip(),

    # Adjusts brightness, contrast, saturation, and hue.
    # Values are slightly reduced from typical defaults to prevent extreme color changes
    # that might make food items unrecognizable.
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),

    # Converts the image to a PyTorch Tensor (HWC to CHW format, and scales pixel values to [0, 1]).
    transforms.ToTensor(),

    # Normalizes the tensor with mean and standard deviation.
    # These are the standard ImageNet statistics. Use them if you're fine-tuning
    # a model pre-trained on ImageNet. If training from scratch, you should
    # calculate the mean/std of your Food-101 dataset.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    # Resize the image to 256x256 (a common practice before center cropping).
    transforms.Resize(256),

    # Crops the central 224x224 portion of the image. This ensures consistent
    # input size for the model during evaluation.
    transforms.CenterCrop(IMAGE_SIZE),

    # Converts the image to a PyTorch Tensor.
    transforms.ToTensor(),

    # Normalizes the tensor using the same ImageNet statistics as training.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("--- Standard Transforms (for 224x224 input) ---")
print("Train Transform:")
print(train_transform)
print("\nTest Transform:")
print(test_transform)

In [ ]:
#create train and test datasets
train_dataset = Food101(data_path,train_path,train_transform)
test_dataset = Food101(data_path,test_path,test_transform)

In [ ]:
len(train_dataset)

In [ ]:
#print a class to check
img,label = train_dataset[0]
img.shape
plt.imshow(img.permute(1,2,0))
plt.title(id_to_class[label])
plt.axis('off')

In [ ]:
#create dataloaders
BATCH_SIZE = 64
train_dataloader = DataLoader(train_dataset,
                             batch_size = BATCH_SIZE,
                             shuffle = True,
                             pin_memory = True
                            )
test_dataloader = DataLoader(test_dataset,
                            batch_size = BATCH_SIZE,
                            shuffle = False,
                            pin_memory=True
                            )

In [ ]:
# import torch
# import torch_xla
# import torch_xla.core.xla_model as xm

# print("PyTorch version:", torch.__version__)
# print("TPU device:", xm.xla_device())
#  # should print something like: xla:1

In [ ]:
#resnet - 50 model
import torch
import torch.nn as nn
import torch.nn.functional as F

class Bottleneck(nn.Module):
    expansion = 4 # Bottleneck blocks expand the channels by 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.downsample = downsample # This will be used for projection shortcut
        self.stride = stride

    def forward(self, x):
        identity = x # Store the input for the skip connection

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        # Apply downsample if needed (for projection shortcut)
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity # Add the shortcut connection
        out = self.relu(out) # Apply ReLU after adding the shortcut

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=101):
        super(ResNet, self).__init__()
        self.in_channels = 64 # Initial input channels for the first residual block

        # Initial layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet stages (conv2_x to conv5_x)
        # Each _make_layer call creates a stack of residual blocks
        self.layer1 = self._make_layer(block, 64, layers[0]) # conv2_x: 3 blocks
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2) # conv3_x: 4 blocks, first block has stride 2
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2) # conv4_x: 6 blocks, first block has stride 2
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2) # conv5_x: 3 blocks, first block has stride 2

        # Final layers
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) # Global Average Pooling
        self.fc = nn.Linear(512 * block.expansion, num_classes) # Final fully connected layer



    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        # Projection shortcut logic:
        # 1. If stride is not 1 (spatial downsampling) OR
        # 2. If input channels don't match expanded output channels
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )

        layers = []
        # Add the first block in the layer (which might have a stride > 1 and a projection shortcut)
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion # Update in_channels for subsequent blocks

        # Add the remaining blocks in the layer (which use identity shortcuts)
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        # Initial layers
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Residual stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Final layers
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # Flatten the (N, C, 1, 1) to (N, C)
        x = self.fc(x)

        return x

def ResNet50(num_classes=101):
    # ResNet-50 uses Bottleneck blocks with [3, 4, 6, 3] blocks in its 4 stages
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)



In [ ]:
# !pip install torchinfo
# from torchinfo import summary
model = ResNet50(101)
model.to(device)
# summary(model,input_size=(3,224,224))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [ ]:
from torch.optim.lr_scheduler import StepLR

scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
#write a function to calculate time
from timeit import default_timer as timer
def find_time(start,end,device):
    seconds = end - start

    minutes = seconds // 60
    rem_sec = seconds % 60

    print(f'training time on {device}: {int(minutes)} minutes {int(rem_sec)} seconds.')

In [ ]:
import torch
from timeit import default_timer as timer 

torch.manual_seed(42)

epochs = 5
train_losses = []
train_accuracies = []

start = timer()

for epoch in range(epochs):
    # --- IMPORTANT: RESET FOR EACH EPOCH ---
    train_loss = 0.0
    correct = 0
    total = 0
    # --- Set the model to training mode ---
    model.train()

    for batch_id, (image, label) in enumerate(train_dataloader):
        image, label = image.to(device), label.to(device)

        # --- Optimization steps (usually at the start of batch processing) ---
        optimizer.zero_grad() # Zero gradients for this batch

        output = model(image) # Forward pass

        loss = criterion(output, label) # Calculate loss

        loss.backward() # Backpropagation
        optimizer.step() # Update model parameters

        # --- Accumulate loss and calculate accuracy for the current batch ---
        train_loss += loss.item() * image.size(0) # Accumulate total loss for the epoch

        y_pred = torch.argmax(output, axis=1) # Get predicted classes
        correct += (y_pred == label).sum().item() # Count correct predictions
        total += label.size(0) # Accumulate total samples processed
    scheduler.step()
    # --- Calculate epoch-level metrics after processing all batches in the epoch ---
    # Divide total accumulated loss by the total number of samples in the dataset
    epoch_train_loss = train_loss / len(train_dataloader.dataset)
    # Calculate accuracy as percentage
    epoch_train_accuracy = 100 * (correct / total)

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)

    # --- Print epoch results ---
    print(f'Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | Train Accuracy: {epoch_train_accuracy:.2f}%')

end = timer()
# Assuming find_time(start, end) prints the time difference
find_time(start, end,device)

In [ ]:
#save the model
# Save just the weights (recommended)
# torch.save(model.state_dict(), 'resnet50_weights.pth')


In [ ]:
import pickle

# Save the entire model (architecture + weights)
with open('resnet50_model-80.pkl', 'wb') as f:
    pickle.dump(model, f)

In [ ]:
!ls -lh

In [ ]:
# Make sure it's in /kaggle/working/
import shutil
shutil.move('resnet50_model-2.pkl', '/kaggle/working/resnet50_model.pkl')

In [ ]:
# #load again
# import torchvision.models as models

# # Load model architecture
# model = models.resnet50(pretrained=False, num_classes=NUM_CLASSES)

# # Load saved weights
# model.load_state_dict(torch.load('resnet50_weights.pth'))

# # Move to device and resume training
# model.to(device)
# model.train()

import pickle

# Load the model (make sure architecture code is available in the script)
with open('/kaggle/input/model-80/resnet50_model-80.pkl', 'rb') as f:
    model = pickle.load(f)
# Move to device and set to eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.eval()


In [ ]:
#testing loop
def test_model(model, test_loader, device):
    model.eval()  # set to evaluation mode
    start = timer()
    correct = 0
    total = 0
    test_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():  # no gradient calculation needed
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = test_loss / len(test_loader)
    accuracy = 100 * correct / total
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")
    # return avg_loss, accuracy
    end = timer()
    find_time(start,end,device)
test_model(model,test_dataloader,device)

In [ ]:
#get top 5 predictions
import os
from PIL import Image
import torch.nn.functional as F

class_names = list(class_to_id.keys())
cls_id = int(input("enter class id for which you want to see top 5 predictions: "))
choosen_name = class_names[cls_id]
print(f"you have chosen to see {class_names[cls_id]}")
dir_path = '/kaggle/input/food41/images'
cls_folder = os.path.join(dir_path,choosen_name)
cls_item = os.listdir(cls_folder)[0]
item_path = os.path.join(cls_folder,cls_item)

image = Image.open(item_path).convert('RGB')
img_Value = test_transform(image).unsqueeze(0).to(device)

def predict_top_5(model,image,class_names):
        with torch.inference_mode():
            image = image.to(device)
            output_logits = model(image)
            probabilities = F.softmax(output_logits,dim=1)

            # torch.topk returns (values, indices)
            top_probs , top_indices = torch.topk(probabilities,5,dim=1)

            #this will be in tensor shape of (1,5) where 1 is the batch_dimension
            #we need to convert this into a list and remove the dimensions

            top_probs = top_probs.squeeze().cpu().numpy().tolist()
            top_indices = top_indices.squeeze().cpu().numpy().tolist()

            top_pred = []
            print("Top 5 predicted classes:")
            for i in range(5):
                print(f" {class_names[top_indices[i]]} - probability: {top_probs[i]}")

predict_top_5(model,img_Value,class_names)

plt.imshow(image)
plt.axis('off')
plt.show()      

In [ ]:
import matplotlib.pyplot as plt

# Plotting training loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()

# Plotting training accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy')
plt.legend()

plt.tight_layout()
plt.show()